#1. Parameters

In [0]:
%run "./CCU063_03-D01-parameters"

#2. Load maternity_interpreter_cohort table

In [0]:
maternity_interpreter_cohort = spark.table(f'{dbc}.{proj}_maternity_interpreter_cohort_prefilter')

maternity_interpreter_cohort.printSchema()
maternity_interpreter_cohort = (maternity_interpreter_cohort
.select('person_id_mother_deid', 'uniqpregid', 'est_preg_start',  'lookback_start', 'lookback_issue_flag', 'NHS_NUMBER_interpreter', 'SNOMED_conceptId', 'SNOMED_conceptId_description', 'DATE_interpreter', 'RECORD_DATE_interpreter', 'interpreter_use', 'record_before_lookback', 'delivery_date'))

###NEED TO SELECT THE APPROPRIATE VARIABLES FROM marternity_interpreter_cohort

In [0]:
count_var(maternity_interpreter_cohort, 'person_id_mother_deid')

#3. Exploring when interpreter code recorded

In [0]:
# Another attempt at adding column for interpreter recorded before preconception period but it includes all women not just those with interpreter use - we need to include just those with interpreter use (also doesn't really make sense if there are multiple interpreter rows per mother)
maternity_interpreter_cohort2 = maternity_interpreter_cohort.filter(f.col("interpreter_use")=="yes").withColumn(
    "record_before_lookback",
    f.when(f.col("DATE_interpreter") < f.col("lookback_start"), "yes").otherwise("no"),
)

In [0]:
#datediff = maternity_interpreter_cohort2.withColumn('daysbeforepreg',f.datediff(f.col('est_preg_start'), f.col('DATE_interpreter')))

In [0]:
tmp1 = ( maternity_interpreter_cohort2
        .withColumn('daysbeforepreg',f.datediff(f.col('est_preg_start'), f.col('DATE_interpreter')))
        .withColumn('weeksbeforepreg', f.col('daysbeforepreg') / -7 ) 
        .withColumn('weeksbeforepreg_rounded', f.round(f.col('weeksbeforepreg')))
        .withColumn('period', 
                    f.when(f.col("DATE_interpreter") < f.col("lookback_start"), "before_lookback")
                    .when(f.col("DATE_interpreter") > f.col("est_preg_start"), "post_conception" )
#                    .when(f.col("DATE_interpreter") > f.col(""), "after_delivery" )
                    .otherwise("during_lookback")
                    )
        )


tmp2= (tmp1
       .groupBy('weeksbeforepreg_rounded', 'period')
        .agg(f.count('person_id_mother_deid').alias('count'))
        .sort(f.col('weeksbeforepreg_rounded'))
        )


#display(tmp1.limit(100 ))
display(tmp2)
tab( tmp1, "period")

Databricks visualization. Run in Databricks to view.

In [0]:
tmp1 = ( maternity_interpreter_cohort2
        .withColumn('daysbeforepreg',f.datediff(f.col('est_preg_start'), f.col('DATE_interpreter')))
        .withColumn('weeksbeforepreg', f.col('daysbeforepreg') / -7 ) 
        .withColumn('weeksbeforepreg_rounded', f.round(f.col('weeksbeforepreg')))
        .withColumn('est_del_date', f.col('est_preg_start') + f.lit(280))
        .withColumn('period', 
                    f.when(f.col("DATE_interpreter") < f.col("lookback_start"), "before_lookback")
                    .when((f.col("DATE_interpreter") > f.col("est_preg_start")) & (f.col("DATE_interpreter") < f.col("est_del_date")), "post_conception")
                    .when(f.col("DATE_interpreter") >= f.col("est_del_date"), "after_delivery")
#                    .when(f.col("DATE_interpreter") > f.col(""), "after_delivery" )
                    .otherwise("during_lookback")
                    ))
        




In [0]:
tmp2= (tmp1
       .groupBy('weeksbeforepreg_rounded', 'period')
        .agg(f.count('person_id_mother_deid').alias('count'))
        .sort(f.col('weeksbeforepreg_rounded'))
        )


#display(tmp1.limit(100 ))
display(tmp2)
tab( tmp1, "period")

In [0]:
tmp1 = ( maternity_interpreter_cohort2
        .withColumn('daysbeforepreg',f.datediff(f.col('est_preg_start'), f.col('DATE_interpreter')))
        .withColumn('weeksbeforepreg', f.col('daysbeforepreg') / -7 ) 
        .withColumn('weeksbeforepreg_rounded', f.round(f.col('weeksbeforepreg')))
        .withColumn('est_del_date', f.col('est_preg_start') + f.lit(280))
        .withColumn('period', 
                    f.when(f.col("DATE_interpreter") < f.col("lookback_start"), "before_lookback")
                    .when(f.col("DATE_interpreter") > f.col("est_preg_start"), "post_conception")
                    .when(f.col("DATE_interpreter") > f.col("delivery_date"), "after_delivery")
#                    .when(f.col("DATE_interpreter") > f.col(""), "after_delivery" )
                    .otherwise("during_lookback")
                    ))

In [0]:
tmp2= (tmp1
       .groupBy('weeksbeforepreg_rounded', 'period')
        .agg(f.count('person_id_mother_deid').alias('count'))
        .sort(f.col('weeksbeforepreg_rounded'))
        )


#display(tmp1.limit(100 ))
display(tmp2)
tab( tmp1, "period")

Databricks visualization. Run in Databricks to view.

In [0]:
tmp1 = ( maternity_interpreter_cohort2
        .withColumn('daysbeforepreg',f.datediff(f.col('est_preg_start'), f.col('DATE_interpreter')))
        .withColumn('weeksbeforepreg', f.col('daysbeforepreg') / -7 ) 
        .withColumn('weeksbeforepreg_rounded', f.round(f.col('weeksbeforepreg')))
        .withColumn('period', 
                    f.when(f.col("DATE_interpreter") < f.col("lookback_start"), "before_lookback")
                    .when(f.col("DATE_interpreter") > f.col("delivery_date"), "after_delivery")
                    .when(f.col("DATE_interpreter") > f.col("est_preg_start"), "post_conception" )
                    .otherwise("during_lookback")
                    )
        )


tmp2= (tmp1
       .groupBy('weeksbeforepreg_rounded', 'period')
        .agg(f.count('person_id_mother_deid').alias('count'))
        .sort(f.col('weeksbeforepreg_rounded'))
        )


#display(tmp1.limit(100 ))
display(tmp2)
print(tab( tmp1, "period"))
display(  tmp1.filter(f.col('period') == 'after_delivery').limit(100) )

Databricks visualization. Run in Databricks to view.

In [0]:
display(tab( tmp1, "period"))

In [0]:

#display(datediff.limit(100))

In [0]:
#tab(datediff, 'daysbeforepreg')